In [1]:
import pandas as pd
import os
import pickle
import bmra_prep
import bmra_prep.pathway_activity.prediction

In [2]:
cell_line ='BC3C_dec'

data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}/00_outputs_2020_{cell_line}/"
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}/01_outputs_2020_{cell_line}/"


os.makedirs(out_dir, exist_ok = True)

# Load Data

In [3]:
# load metdadata dict and extract used elements
with open(os.path.join(data_dir, "metadata.pickle"), "rb") as f:
    all_metadata = pickle.load(f)

n_modules = all_metadata["n_modules"]
n_genes = all_metadata["n_genes"]
n_experiments = all_metadata["n_experiments"]

modules = all_metadata["modules"]
exp_ids = all_metadata["exp_ids"]
genes = all_metadata["genes"]

In [4]:
# load data
L1000_df = pd.read_csv(
    os.path.join(data_dir, "L1000_Data_norm_data.csv"),
    index_col = 0,
)

x = L1000_df.values
x.shape

(978, 106)

In [5]:
# load doses and perturbation matrix
inhib_conc_matrix = pd.read_csv(
    os.path.join(data_dir, "inhib_conc_annotated.csv"),
    index_col = 0,
).values

ic50_matrix = pd.read_csv(
    os.path.join(data_dir, "ic50_annotated.csv"),
    index_col = 0,
).values

# gamma_matrix = pd.read_csv(
#     os.path.join(data_dir, "gamma_annotated.csv"),
#     index_col = 0,
# ).values

pert_matrix = pd.read_csv(
    os.path.join(data_dir, "pert_annotated.csv"),
    index_col = 0,
).values

In [6]:
# y_true = (1 + gamma_matrix * inhib_conc_matrix / ic50_matrix) / (1 + inhib_conc_matrix / ic50_matrix)

y_true = 1 / (1 + inhib_conc_matrix / ic50_matrix)

display(y_true.shape)
y_true

(10, 106)

array([[1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       ...,
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 0.41176471, 0.41176471,
        0.41176471]])

## Run models

In [13]:
a_coeffs = bmra_prep.pathway_activity.prediction.predict_coeffs(
    x, y_true, pert_matrix, 200_000, 10, 10, 5, 100)

In [14]:
a_coeffs_df = pd.DataFrame(a_coeffs, index = modules, columns = genes)
a_coeffs_df.to_csv(os.path.join(out_dir, "a_coeffs.csv"))
#a_coeffs_df = pd.read_csv(os.path.join(out_dir,'a_coeffs.csv'),index_col=0)
#a_coeffs = a_coeffs_df.values
trh = 0.0001
display((abs(a_coeffs_df) >trh).sum(axis = "columns"))
display(a_coeffs_df)

CDK1_2      14
CDK4_6       8
EGFR        15
Estrogen    12
FGFR        17
PI3K        36
p53         27
TOP2A       11
Src          7
SMAD3        1
dtype: int64

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
CDK1_2,0.000011,0.000006,0.000030,-0.000004,-0.000010,-1.266711e-05,-0.000004,-5.022153e-06,1.211364e-05,-0.000002,...,-4.881537e-06,-0.000012,8.381710e-07,-0.000029,-2.433839e-06,0.000005,0.000023,0.000013,0.000011,-0.000004
CDK4_6,0.000013,0.000001,0.000003,0.000001,-0.000012,8.096357e-05,-0.000005,-1.244571e-06,-9.858684e-07,-0.000004,...,4.393983e-06,-0.000011,-5.412417e-06,0.000017,1.126226e-05,-0.000014,-0.000004,-0.000008,-0.000009,0.000017
EGFR,-0.000009,-0.000004,0.000004,-0.000001,-0.000006,-4.103523e-04,0.000012,9.323190e-06,7.636811e-06,-0.023161,...,-5.122485e-06,0.000025,8.923863e-06,-0.000029,2.789332e-05,0.000015,0.000007,0.000014,0.000002,-0.000007
Estrogen,-0.000027,-0.000002,0.000007,0.000008,0.000053,1.750532e-05,-0.000017,-1.987006e-05,9.008229e-06,-0.299296,...,-5.703575e-06,0.000010,1.310457e-05,0.000004,6.243772e-06,-0.000002,-0.000036,-0.000015,0.000002,-0.000006
FGFR,-0.001038,-0.000005,-0.000004,0.000006,0.000004,-1.631520e-05,-0.000018,2.582136e-06,8.284209e-06,-0.001870,...,-3.719251e-06,-0.000023,-3.568825e-06,0.000005,1.718872e-05,0.000119,-0.000008,0.000041,-0.000014,0.000024
PI3K,0.000006,0.000024,0.000007,-0.000002,-0.000016,-5.309029e-06,0.000002,1.216825e-05,1.691831e-06,-0.000002,...,-1.182727e-06,0.000013,-3.192131e-06,-0.000005,-4.756872e-07,-0.000017,0.000008,0.000008,-0.000009,0.000026
p53,0.000015,0.000013,0.000027,0.000030,0.000004,1.813321e-05,0.260002,-4.622532e-06,9.094177e-06,-0.000005,...,4.499995e-07,-0.000014,-1.325630e-05,0.000016,2.948772e-05,-0.000016,-0.000018,-0.000019,0.000034,0.000036
TOP2A,0.000007,0.000004,-0.000002,-0.000019,-0.000015,1.431598e-05,0.000022,-5.208495e-05,-1.787787e-05,-0.000003,...,-3.897070e-06,0.000002,-1.354778e-05,-0.000050,2.781321e-05,0.000044,0.000002,-0.000009,-0.000017,-0.000080
Src,0.000002,0.000007,0.000013,-0.000024,0.000006,7.109872e-07,-0.000002,5.420715e-07,1.128069e-05,-0.000002,...,9.407705e-06,0.000004,-2.005945e-06,-0.000001,2.836117e-06,-0.000005,-0.000007,-0.000017,0.000029,-0.000011
SMAD3,0.000012,0.000012,0.000010,0.000025,-0.000006,-3.323619e-05,-0.000017,-3.737606e-05,-1.583534e-05,-0.000018,...,-5.788710e-06,-0.000009,-1.568243e-06,-0.000028,1.493956e-05,0.000016,-0.000008,-0.000010,0.000017,0.000002


In [15]:
#pathway_activity = a_coeffs @ x
#pathway_activity.shape

In [16]:
R_global = bmra_prep.pathway_activity.calc_global_response_from_pathway_activity(
    bmra_prep.pathway_activity.calc_pathway_activity(x,a_coeffs),
    modules,
    L1000_df.columns
)
R_global_df = R_global.dataframe
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,-0.759337,-0.552893,0.035743,0.020738,0.052145,-0.238840,-0.078586,-0.395800,0.155988,-0.009729,...,-0.377878,-0.354942,-0.326443,-0.341349,-0.316623,-0.329482,-0.323135,-0.351756,-0.352108,-0.376721
CDK4_6,0.004397,-0.089177,-0.280625,-0.119552,0.028854,-0.110051,-0.095499,-0.045004,-0.255722,0.009669,...,0.242802,0.237798,0.207971,0.187998,0.172693,0.183798,0.212493,0.202204,0.237615,0.206721
EGFR,0.603534,0.513122,0.221285,0.381010,0.550584,0.128331,-0.446104,0.263245,0.283416,-0.023016,...,-1.336891,-0.871765,-0.860751,-0.740957,-0.712548,-0.728234,-0.786046,-0.961295,-0.937902,-0.993657
Estrogen,-0.192528,-0.298931,-0.285204,-0.502063,-0.988345,-0.325069,-0.088520,-0.358271,-0.237663,-0.053975,...,-0.290522,-0.266001,-0.240539,-0.266603,-0.251130,-0.257604,-0.230746,-0.277320,-0.258642,-0.303482
FGFR,-0.113130,-0.196413,-0.099669,0.065474,-0.034717,-0.436489,-0.028167,-0.065509,-0.073538,-0.295952,...,-0.904925,-0.739046,-0.677600,-0.680026,-0.608209,-0.660743,-0.665450,-0.753880,-0.762342,-0.839164
PI3K,-1.943895,-1.863624,-1.682186,-1.419462,-0.621484,-0.177640,-0.008272,-0.483758,-1.375115,-0.237255,...,-0.132020,-0.075732,-0.030197,-0.066946,-0.040962,-0.082623,-0.053640,-0.085105,-0.085941,-0.166477
p53,-0.262849,-0.328878,-0.085118,-0.341063,0.031426,-1.805482,-1.719131,-0.118219,-0.070057,-1.551187,...,0.056287,0.035891,0.048544,0.051309,0.023335,0.051762,0.073851,0.032916,0.075177,0.039925
TOP2A,-0.292842,0.071317,-0.202687,-0.184436,-0.133276,0.019082,0.012915,-1.998258,-0.251282,-0.389292,...,0.153661,0.143817,0.135635,0.113246,0.102625,0.130764,0.141249,0.127625,0.152532,0.145638
Src,-1.055460,-1.990811,0.580747,-1.501162,0.615131,-1.374638,0.537907,0.456359,0.457399,0.499039,...,-0.111272,-0.141251,-0.086139,-0.115508,-0.045002,-0.106340,-0.086825,-0.100296,-0.142246,-0.152101
SMAD3,0.021151,-0.116255,0.031533,-0.348121,0.057626,0.081111,0.026304,0.136905,0.140040,0.061714,...,-0.898236,-0.822701,-0.785937,-0.814870,-0.780934,-0.790786,-0.776757,-0.841006,-0.825758,-0.878606


In [17]:
R_global_df.to_csv(os.path.join(out_dir, "R_global_annotated.csv"))
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,-0.759337,-0.552893,0.035743,0.020738,0.052145,-0.238840,-0.078586,-0.395800,0.155988,-0.009729,...,-0.377878,-0.354942,-0.326443,-0.341349,-0.316623,-0.329482,-0.323135,-0.351756,-0.352108,-0.376721
CDK4_6,0.004397,-0.089177,-0.280625,-0.119552,0.028854,-0.110051,-0.095499,-0.045004,-0.255722,0.009669,...,0.242802,0.237798,0.207971,0.187998,0.172693,0.183798,0.212493,0.202204,0.237615,0.206721
EGFR,0.603534,0.513122,0.221285,0.381010,0.550584,0.128331,-0.446104,0.263245,0.283416,-0.023016,...,-1.336891,-0.871765,-0.860751,-0.740957,-0.712548,-0.728234,-0.786046,-0.961295,-0.937902,-0.993657
Estrogen,-0.192528,-0.298931,-0.285204,-0.502063,-0.988345,-0.325069,-0.088520,-0.358271,-0.237663,-0.053975,...,-0.290522,-0.266001,-0.240539,-0.266603,-0.251130,-0.257604,-0.230746,-0.277320,-0.258642,-0.303482
FGFR,-0.113130,-0.196413,-0.099669,0.065474,-0.034717,-0.436489,-0.028167,-0.065509,-0.073538,-0.295952,...,-0.904925,-0.739046,-0.677600,-0.680026,-0.608209,-0.660743,-0.665450,-0.753880,-0.762342,-0.839164
PI3K,-1.943895,-1.863624,-1.682186,-1.419462,-0.621484,-0.177640,-0.008272,-0.483758,-1.375115,-0.237255,...,-0.132020,-0.075732,-0.030197,-0.066946,-0.040962,-0.082623,-0.053640,-0.085105,-0.085941,-0.166477
p53,-0.262849,-0.328878,-0.085118,-0.341063,0.031426,-1.805482,-1.719131,-0.118219,-0.070057,-1.551187,...,0.056287,0.035891,0.048544,0.051309,0.023335,0.051762,0.073851,0.032916,0.075177,0.039925
TOP2A,-0.292842,0.071317,-0.202687,-0.184436,-0.133276,0.019082,0.012915,-1.998258,-0.251282,-0.389292,...,0.153661,0.143817,0.135635,0.113246,0.102625,0.130764,0.141249,0.127625,0.152532,0.145638
Src,-1.055460,-1.990811,0.580747,-1.501162,0.615131,-1.374638,0.537907,0.456359,0.457399,0.499039,...,-0.111272,-0.141251,-0.086139,-0.115508,-0.045002,-0.106340,-0.086825,-0.100296,-0.142246,-0.152101
SMAD3,0.021151,-0.116255,0.031533,-0.348121,0.057626,0.081111,0.026304,0.136905,0.140040,0.061714,...,-0.898236,-0.822701,-0.785937,-0.814870,-0.780934,-0.790786,-0.776757,-0.841006,-0.825758,-0.878606
